In [ ]:
# CELL 1 — imports (no pip here, handled by permanent cell)
import pandas as pd
import numpy as np
import pandas_ta as ta
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('Imports done')

In [ ]:
# CELL 2 — Fetch BTC/USDT data via yfinance (works from anywhere)
import yfinance as yf

raw = yf.download('BTC-USD', period='60d', interval='1h', progress=False, auto_adjust=True)
raw.columns = [c[0].lower() if isinstance(c, tuple) else c.lower() for c in raw.columns]
df = raw[['open','high','low','close','volume']].copy()
df = df.dropna()

print(f'Fetched {len(df)} candles | BTC-USD 1h')
print(f'From : {df.index[0]}')
print(f'To   : {df.index[-1]}')
print(f'Price: ${df.close.iloc[-1]:,.2f}')
df.tail(3)

In [ ]:
# CELL 3 — Strategy 1: EMA Crossover + RSI
from enum import Enum

class Signal(Enum):
    BUY  = 'BUY'
    SELL = 'SELL'
    HOLD = 'HOLD'

def ema_crossover_signals(df):
    df = df.copy()
    df['ema_fast']   = ta.ema(df['close'], length=9)
    df['ema_slow']   = ta.ema(df['close'], length=21)
    df['rsi']        = ta.rsi(df['close'], length=14)
    df['atr']        = ta.atr(df['high'], df['low'], df['close'], length=14)
    df['cross_up']   = (df['ema_fast'] > df['ema_slow']) & (df['ema_fast'].shift(1) <= df['ema_slow'].shift(1))
    df['cross_down'] = (df['ema_fast'] < df['ema_slow']) & (df['ema_fast'].shift(1) >= df['ema_slow'].shift(1))
    return df

print('Strategy 1 (EMA Crossover) loaded')

In [ ]:
# CELL 4 — Strategy 2: Bollinger Band Mean Reversion
def mean_reversion_signals(df):
    df  = df.copy()
    bb  = ta.bbands(df['close'], length=20, std=2.0)
    col = [c for c in bb.columns]
    df['bb_upper']  = bb[col[2]]
    df['bb_middle'] = bb[col[1]]
    df['bb_lower']  = bb[col[0]]
    df['bb_width']  = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']
    df['rsi']       = ta.rsi(df['close'], length=14)
    df['atr']       = ta.atr(df['high'], df['low'], df['close'], length=14)
    return df

print('Strategy 2 (Mean Reversion) loaded')

In [ ]:
# CELL 5 — Backtest Engine (0.1% fee + 30% Indian tax on profits)
def run_backtest(df, strategy='ema', capital=1000, risk_pct=0.02,
                 commission=0.001, tax_rate=0.30):
    df = ema_crossover_signals(df) if strategy == 'ema' else mean_reversion_signals(df)
    trades, equity, cash = [], [capital], capital
    in_trade, trade = False, None

    for i in range(50, len(df)):
        row = df.iloc[i]
        atr = row.get('atr', row['close'] * 0.02)
        if pd.isna(atr): atr = row['close'] * 0.02

        if in_trade and trade:
            hit_sl = hit_tp = False
            if trade['side'] == 'BUY':
                hit_sl = row['low']  <= trade['sl']
                hit_tp = row['high'] >= trade['tp']
            else:
                hit_sl = row['high'] >= trade['sl']
                hit_tp = row['low']  <= trade['tp']

            if hit_tp or hit_sl:
                exit_px   = trade['tp'] if hit_tp else trade['sl']
                gross_pnl = (exit_px - trade['entry']) * trade['qty'] if trade['side'] == 'BUY' \
                            else (trade['entry'] - exit_px) * trade['qty']
                fee     = exit_px * trade['qty'] * commission
                tax     = gross_pnl * tax_rate if gross_pnl > 0 else 0
                net_pnl = gross_pnl - fee - tax
                cash   += net_pnl
                trades.append({'side': trade['side'], 'entry': trade['entry'],
                               'exit': exit_px, 'reason': 'TP' if hit_tp else 'SL',
                               'gross_pnl': gross_pnl, 'tax': tax, 'net_pnl': net_pnl})
                in_trade, trade = False, None

        if not in_trade:
            entry = row['close']; signal = Signal.HOLD; sl = tp = 0
            if strategy == 'ema':
                rsi = row.get('rsi', 50)
                if pd.isna(rsi): rsi = 50
                if row.get('cross_up', False) and rsi < 60:
                    signal = Signal.BUY;  sl = entry - atr*2; tp = entry + (entry-sl)*2
                elif row.get('cross_down', False) and rsi > 40:
                    signal = Signal.SELL; sl = entry + atr*2; tp = entry - (sl-entry)*2
            else:
                rsi    = row.get('rsi', 50)
                bb_low = row.get('bb_lower', 0)
                bb_up  = row.get('bb_upper', 9e9)
                bb_mid = row.get('bb_middle', entry)
                if not any(pd.isna([rsi, bb_low, bb_up, bb_mid])):
                    if row['close'] <= bb_low and rsi < 30:
                        signal = Signal.BUY;  sl = entry - atr*1.5; tp = bb_mid
                    elif row['close'] >= bb_up and rsi > 70:
                        signal = Signal.SELL; sl = entry + atr*1.5; tp = bb_mid

            if signal != Signal.HOLD and abs(entry - sl) > 0:
                qty = min((cash*risk_pct)/abs(entry-sl), (cash*0.95)/entry)
                cash -= entry * qty * commission
                in_trade = True
                trade = {'side': signal.value, 'entry': entry, 'sl': sl, 'tp': tp, 'qty': qty}

        equity.append(cash)

    return pd.DataFrame(trades), pd.Series(equity)

print('Backtest engine loaded')

In [ ]:
# CELL 6 — Run + compare both strategies
CAPITAL = 1000

ema_trades, ema_equity = run_backtest(df, strategy='ema', capital=CAPITAL)
mr_trades,  mr_equity  = run_backtest(df, strategy='mr',  capital=CAPITAL)

def summarize(name, trades, equity):
    if len(trades) == 0:
        print(f'\n{name}: No trades generated'); return None
    wins   = trades[trades['net_pnl'] > 0]
    losses = trades[trades['net_pnl'] <= 0]
    final  = equity.iloc[-1]
    ret    = (final - CAPITAL) / CAPITAL * 100
    wr     = len(wins) / len(trades) * 100
    pf     = wins['net_pnl'].sum() / abs(losses['net_pnl'].sum()) if len(losses) > 0 else 999
    dd     = ((equity - equity.cummax()) / equity.cummax()).min() * 100
    print(f'''
============================================
  {name}
============================================
  Capital Start  : ${CAPITAL:,.2f}
  Capital End    : ${final:,.2f}
  Total Return   : {ret:+.2f}%
  Max Drawdown   : {dd:.2f}%
  Profit Factor  : {pf:.2f}
  Total Trades   : {len(trades)}
  Win Rate       : {wr:.1f}%
  Avg Win        : ${wins["net_pnl"].mean():.2f}
  Avg Loss       : ${losses["net_pnl"].mean():.2f}
  Tax Paid (30%) : ${trades["tax"].sum():.2f}
============================================''')
    return ret

r1 = summarize('EMA CROSSOVER', ema_trades, ema_equity)
r2 = summarize('MEAN REVERSION', mr_trades, mr_equity)

if r1 is not None and r2 is not None:
    print(f'\n  WINNER: {"EMA CROSSOVER" if r1 > r2 else "MEAN REVERSION"}')

In [ ]:
# CELL 7 — Plot equity curves
plt.figure(figsize=(14,5))
plt.plot(ema_equity.values, label='EMA Crossover', color='blue')
plt.plot(mr_equity.values,  label='Mean Reversion', color='orange')
plt.axhline(y=CAPITAL, color='gray', linestyle='--', label='Starting Capital')
plt.title('BTC-USD 1h — Last 60 days')
plt.xlabel('Candles'); plt.ylabel('Capital ($)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()